# Workshop 3: Segmentación de clientes con Clustering (K-Means vs. Agglomerative vs. DBSCAN)

## 🎯 Objetivo

Segmentar a los clientes de un centro comercial según su comportamiento de compra, aplicando y comparando **tres estrategias de clustering no supervisado**:

- **K-Means**
- **Clustering jerárquico aglomerativo**
- **DBSCAN**

> **Pregunta de negocio:** ¿En cuántos grupos "naturales" se dividen los clientes de este centro comercial, y qué caracteriza a cada uno?

Al final, los grupos descubiertos se convierten en la variable objetivo de un clasificador: **no supervisado → supervisado**.

**Integrantes del equipo:**
- Juan José Escobar Saldarriaga
- Jeremías Figueroa García
- David Quintero Gallego

## ⏱️ Duración y ritmo sugerido (90 min)

| Fase | Contenido | Tiempo sugerido |
|---|---|---:|
| 0 | Carga y preprocesamiento | 10 min |
| 1 | K-Means | 15 min |
| 2 | Agglomerative Clustering | 10 min |
| 3 | DBSCAN | 15 min |
| 4 | Visualización PCA 3D | 10 min |
| 5 | Comparación con Silhouette Score | 5 min |
| 6 | Definir las clases naturales | 5 min |
| 7 | Interpretación de los clusters (perfilado) | 10 min |
| 8-10 | Etiquetado + clasificador + evaluación | 10 min |

## 1. El dataset

**Mall Customer Segmentation Data** (Kaggle): https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python

El archivo `Mall_Customers.csv` contiene 200 clientes:

| Columna | Descripción |
|---|---|
| `CustomerID` | Identificador único del cliente (no es una característica, es un índice) |
| `Genre` | Género del cliente (**ojo**: la columna se llama literalmente `Genre`, no `Gender`) |
| `Age` | Edad del cliente |
| `Annual Income (k$)` | Ingreso anual en miles de dólares |
| `Spending Score (1-100)` | Puntaje de gasto (1 = gasta poco, 100 = gasta mucho) |

In [2]:
#import kagglehub

path = "data/"
print(path)

data/


---
## 2. Fase 0 — Carga y preprocesamiento de datos

### 2.1. Exploración inicial

In [5]:
import os
import pandas as pd

# Si descargaron con kagglehub, el CSV está dentro de `path`.
# Si lo tienen local, usen simplemente "Mall_Customers.csv".
df = pd.read_csv(os.path.join(path, "Mall_Customers.csv"))

display(df.head())
print(f"Shape del dataset: {df.shape}")

print("\nVerificación de valores nulos:")
display(df.isnull().sum())

,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


Shape del dataset: (200, 5)

Verificación de valores nulos:


CustomerID                0
Gender                    0
Age                       0
Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64

In [ ]:
# TODO: Añadan otros pasos de exploración que consideren relevantes
# (ej. df.info(), df.describe(), distribución de Genre, histogramas, pairplot...)

### Preguntas

- ¿Hay valores nulos?
- ¿Qué tipo de dato tiene cada columna? ¿Cuáles son numéricas y cuál es categórica?

**Respuesta:** _Escriban aquí su respuesta._

### 2.2. Selección de variables

`CustomerID` no aporta información para el clustering — es solo un identificador. Elimínenla.

Se usarán **las 4 variables restantes**: `Genre`, `Age`, `Annual Income (k$)` y `Spending Score (1-100)`.

In [ ]:
# TODO: Eliminen la columna CustomerID

### 2.3. Codificación de la variable categórica

`Genre` es categórica (Male/Female) y los algoritmos de clustering trabajan con distancias numéricas.

> **Pista:** ¿`Genre` tiene un orden natural? ¿Cuántas categorías tiene? Recuerden `LabelEncoder`, `OneHotEncoder` o mapear manualmente con `.map({...})` dado que solo son 2 categorías.
>
> La columna resultante debe llamarse `Genre_encoded` (se usa más adelante).

In [ ]:
# from sklearn.preprocessing import ...

# TODO: Completen: codifiquen la columna Genre → df["Genre_encoded"]

### 2.4. Estandarización

In [ ]:
from sklearn.preprocessing import StandardScaler

features = ["Genre_encoded", "Age", "Annual Income (k$)", "Spending Score (1-100)"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

### Pregunta

- ¿Por qué es importante estandarizar antes de aplicar K-Means, Agglomerative Clustering o DBSCAN? (Piensen en qué pasaría si NO estandarizaran — comparen las escalas de `Age` vs. `Annual Income (k$)`).

**Respuesta:** _Escriban aquí su respuesta._

---
## 3. Fase 1 — K-Means

K-Means necesita que le digan de antemano **cuántos clusters ($k$)** formar, y va iterando hasta encontrar $k$ centroides que minimizan la distancia de cada punto a su centroide más cercano.

In [ ]:
from sklearn.cluster import KMeans

k = ...  # <- completen

model = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

labels = model.fit_predict(X_scaled)

### 3.1. Método del codo (Elbow Method)

Graficar la **inercia** (`model.inertia_`) contra distintos valores de $k$ y buscar el "codo" de la curva:

$$
\text{Inercia} = \sum_{i=1}^{n} \left\lVert x_i - \mu_{c(i)} \right\rVert^2
$$

donde $\mu_{c(i)}$ es el centroide del cluster asignado al punto $x_i$.

Prueben, como mínimo: $k = 2, 3, 4, 5, 6, 7, 8$

In [ ]:
import matplotlib.pyplot as plt

inertias = []
ks = range(2, 9)

for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

# TODO: grafiquen ks vs. inertias con matplotlib

### 3.2. Silhouette Score para cada k

In [ ]:
from sklearn.metrics import silhouette_score

# TODO: calculen el Silhouette Score para cada k y construyan la tabla comparativa
# score = silhouette_score(X_scaled, labels)

| k | Inercia | Silhouette Score |
|---:|---:|---:|
| 2 | ... | ... |
| 3 | ... | ... |
| 4 | ... | ... |
| 5 | ... | ... |
| 6 | ... | ... |
| 7 | ... | ... |
| 8 | ... | ... |

### Preguntas

1. ¿El codo de la curva de inercia coincide con el `k` de mayor Silhouette Score?
2. ¿Qué `k` seleccionarían para K-Means? Justifiquen.

**Respuesta:** _Escriban aquí su respuesta._

In [ ]:
# TODO: entrenen K-Means con el k elegido y guarden las etiquetas
labels_kmeans = ...

---
## 4. Fase 2 — Clustering jerárquico aglomerativo

Repitan el mismo flujo del ejercicio de Iris.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z)
plt.show()

In [ ]:
# TODO: prueben k = 2..8 con linkage="ward" y calculen el Silhouette Score de cada uno

| k | Silhouette Score |
|---:|---:|
| 2 | ... |
| 3 | ... |
| 4 | ... |
| 5 | ... |
| 6 | ... |
| 7 | ... |
| 8 | ... |

### Preguntas

- ¿Cuántos grupos sugiere el dendrograma visualmente?
- ¿Coincide con el mejor `k` según Silhouette Score?

**Respuesta:** _Escriban aquí su respuesta._

In [ ]:
# TODO: entrenen Agglomerative con el k elegido y guarden las etiquetas
labels_agglomerative = ...

---
## 5. Fase 3 — DBSCAN

DBSCAN no necesita `k` — el número de clusters emerge de `eps` y `min_samples`.

Combinaciones sugeridas:
- `eps = 0.3, 0.5, 0.7, 0.9, 1.1`
- `min_samples = 3, 5, 8, 10`

Busquen una región de parámetros razonable (ni 1 solo cluster gigante, ni 190 puntos marcados como ruido).

In [ ]:
from sklearn.cluster import DBSCAN

model = DBSCAN(
    eps=...,          # <- exploren varios valores
    min_samples=...   # <- exploren varios valores
)

labels = model.fit_predict(X_scaled)

In [ ]:
# TODO: exploren combinaciones de eps y min_samples
# (n° de clusters, n° de puntos de ruido y Silhouette Score de cada una)

Para el Silhouette Score, recuerden **excluir el ruido** (`label = -1`):

In [ ]:
mask = labels != -1
score = silhouette_score(X_scaled[mask], labels[mask])

| eps | min_samples | N° de clusters | N° de puntos de ruido | Silhouette Score |
|---:|---:|---:|---:|---:|
| ... | ... | ... | ... | ... |

### Preguntas

1. ¿Cuántos clusters encontró DBSCAN con los parámetros que eligieron?
2. ¿Cuántos puntos quedaron como ruido?
3. ¿Qué pasa con el número de clusters si aumentan `eps`? ¿Y si aumentan `min_samples`?

**Respuesta:** _Escriban aquí su respuesta._

In [ ]:
# TODO: entrenen DBSCAN con los parámetros elegidos y guarden las etiquetas
labels_dbscan = ...

---
## 6. Fase 4 — Visualización con PCA en 3D

Con 4 variables no se puede graficar directamente; se reduce a **3 componentes principales** para comparar los métodos en un mismo espacio.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print("Varianza explicada por cada componente:", pca.explained_variance_ratio_)
print("Varianza explicada acumulada:", pca.explained_variance_ratio_.sum())

### Pregunta

- ¿Qué porcentaje de la varianza total logran capturar las 3 componentes? ¿Les parece suficiente para confiar en la visualización?

**Respuesta:** _Escriban aquí su respuesta._

In [ ]:
def plot_clusters_3d(X_pca, labels, title):
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")

    scatter = ax.scatter(
        X_pca[:, 0], X_pca[:, 1], X_pca[:, 2],
        c=labels,
        cmap="viridis",
        s=40
    )

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_zlabel("PC3")
    ax.set_title(title)

    legend = ax.legend(*scatter.legend_elements(), title="Cluster")
    ax.add_artist(legend)

    plt.show()

In [ ]:
# TODO: generen un gráfico por cada método
# plot_clusters_3d(X_pca, labels_kmeans, "K-Means")
# plot_clusters_3d(X_pca, labels_agglomerative, "Agglomerative Clustering")
# plot_clusters_3d(X_pca, labels_dbscan, "DBSCAN")

> **Tip:** para DBSCAN, los puntos de ruido (`-1`) también se colorean — ¿eso dificulta la lectura del gráfico?

**Análisis de las visualizaciones:** _Escriban aquí su análisis._

---
## 7. Fase 5 — Comparación final con Silhouette Score

| Método | Parámetros | N° de clusters | N° de puntos de ruido | Silhouette Score |
|---|---|---:|---:|---:|
| K-Means | `k = ...` | ... | N/A | ... |
| Agglomerative | `k = ..., linkage="ward"` | ... | N/A | ... |
| DBSCAN | `eps = ..., min_samples = ...` | ... | ... | ... |

---
## 8. Fase 6 — ¿Cuáles son las clases naturales?

> **Basándose en su análisis comparativo (Silhouette Score, número de clusters, cantidad de ruido y las visualizaciones 3D), ¿cuántos grupos "naturales" consideran que existen en este dataset de clientes, y qué método usarían para definirlos?**

No necesariamente gana el método con el Silhouette Score más alto: justifiquen considerando también la interpretabilidad y la forma de los grupos.

**Respuesta:** _Escriban aquí su respuesta._

**A partir de aquí se trabaja con el método y el número de clusters seleccionados.**

---
## 9. Fase 7 — Interpretación de los clusters (perfilado)

Se perfila cada cluster con las variables **originales sin escalar** (años, dólares, puntaje).

### 9.1. Agreguen la etiqueta de cluster al DataFrame original

In [ ]:
df["Cluster"] = ...  # las labels del método que seleccionaron en la Fase 6

### 9.2. Métricas por cluster

In [ ]:
display(df.groupby("Cluster")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean())
display(df.groupby("Cluster")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].median())
display(df.groupby("Cluster")["Genre"].value_counts(normalize=True))

### 9.3. Boxplots por variable, separados por cluster

In [ ]:
import seaborn as sns

for col in ["Age", "Annual Income (k$)", "Spending Score (1-100)"]:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df, x="Cluster", y=col)
    plt.title(f"{col} por cluster")
    plt.show()

### 9.4. "Perfil" de cada cluster

Describan cada cluster en una frase. Ajusten el número de filas al número real de clusters.

> **Importante:** la interpretación debe salir de los números, no al revés.

| Cluster | Edad promedio | Ingreso promedio | Spending Score promedio | Interpretación / "persona" |
|---:|---:|---:|---:|---|
| 0 | ... | ... | ... | ... |
| 1 | ... | ... | ... | ... |

### Pregunta

- ¿Alguno de los clusters les llamó la atención por combinar valores que no esperaban (ej. ingreso alto pero gasto bajo)? ¿Qué estrategia de negocio le darían a ese grupo?

**Respuesta:** _Escriban aquí su respuesta._

---
## 10. Fase 8 — El cluster como variable objetivo

Se usa la columna `Cluster` como **target** de un problema de clasificación supervisada, para asignar el segmento de un cliente nuevo sin re-ejecutar el clustering.

In [ ]:
X = df[["Genre_encoded", "Age", "Annual Income (k$)", "Spending Score (1-100)"]]
y = df["Cluster"]

---
## 11. Fase 9 — Entrenen un clasificador

Elijan **uno**: KNN, Decision Tree, Random Forest o SVM.

> **Nota:** `Cluster` es **categórica** — no usen Regresión Lineal. Si quieren algo lineal, usen **Regresión Logística**.

**Clasificador elegido:** _..._

### 11.1. División train/test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # <- piensen: ¿por qué conviene usar stratify aquí?
)

**¿Por qué conviene usar `stratify` aquí?**

**Respuesta:** _Escriban aquí su respuesta._

### 11.2. Entrenamiento

Ejemplo con KNN — adapten según el modelo elegido. Los modelos basados en distancias (KNN, SVM) necesitan escalado; los de árboles no lo requieren, pero no les hace daño.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(n_neighbors=...))  # <- completen / cambien el modelo
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

---
## 12. Fase 10 — Evaluación del clasificador

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

### Preguntas de cierre

1. ¿Qué tan bien logró el clasificador predecir el cluster de un cliente nuevo?
2. Si el accuracy fue muy alto (cercano a 1.0), ¿qué les dice eso sobre qué tan "separables" son los clusters en el espacio de características originales?
3. Si el clasificador se equivocó mucho entre dos clusters en particular, revisen la matriz de confusión: ¿esos dos clusters tenían perfiles parecidos en la Fase 7? ¿Tiene sentido que el modelo los confunda?

**Respuesta:** _Escriban aquí su respuesta._

---
## 13. Reflexión final

(3-4 líneas por pregunta)

> **¿El método de clustering que obtuvo el mayor Silhouette Score fue el que terminaron usando para definir los segmentos de negocio? ¿Por qué sí o por qué no?**

**Respuesta:** _Escriban aquí su respuesta._

> **Si el centro comercial les pidiera, en una sola frase por segmento, una recomendación de marketing para cada cluster, ¿qué les dirían?**

**Respuesta:** _Escriban aquí su respuesta._